In [ ]:
import os
import base64
import faiss
import torch
import ollama
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer, util

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
ALPHA = 0.4          # Weight for visual score (1-ALPHA goes to semantic score)
OLLAMA_MODEL = "phi4-mini"   # Must be pulled: `ollama pull phi4-mini`

MODELS = {}


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def get_raw_tensor(output):
    """Deep-access helper to extract the actual tensor from any HF output."""
    if torch.is_tensor(output):
        return output
    for attr in ("image_embeds", "pooler_output", "last_hidden_state"):
        if hasattr(output, attr):
            return getattr(output, attr)
    try:
        return output[0]
    except Exception:
        return output


def image_to_base64(path: str) -> str:
    """Return a base64-encoded string for a local image file."""
    with open(path, "rb") as fh:
        return base64.b64encode(fh.read()).decode("utf-8")


# ---------------------------------------------------------------------------
# Stack Initialization
# ---------------------------------------------------------------------------

def initialize_stack():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"--- [INFO] Initializing Stack on: {device} ---")

    # 1. CLIP  (Stage 1 – visual similarity)
    clip_id = "openai/clip-vit-base-patch32"
    MODELS["clip_model"] = CLIPModel.from_pretrained(clip_id).to(device)
    MODELS["clip_processor"] = CLIPProcessor.from_pretrained(clip_id)

    # 2. SBERT  (Stage 3 – semantic text alignment)
    MODELS["text_model"] = SentenceTransformer("all-MiniLM-L6-v2")

    # 3. Detect CLIP embedding dimension dynamically
    dummy = torch.randn(1, 3, 224, 224).to(device)
    with torch.no_grad():
        raw_out = MODELS["clip_model"].get_image_features(pixel_values=dummy)
        dummy_tensor = get_raw_tensor(raw_out)

    MODELS["dim"] = dummy_tensor.shape[-1]
    MODELS["device"] = device
    print(f"--- [SUCCESS] Models ready. CLIP dim: {MODELS['dim']} ---")


initialize_stack()


# ---------------------------------------------------------------------------
# Stage 1 – CLIP visual features
# ---------------------------------------------------------------------------

def get_clip_features(path: str) -> np.ndarray:
    """Return an L2-normalised (1, dim) float32 CLIP embedding."""
    device = MODELS["device"]
    img = Image.open(path).convert("RGB")
    inputs = MODELS["clip_processor"](images=img, return_tensors="pt").to(device)

    with torch.no_grad():
        raw_out = MODELS["clip_model"].get_image_features(**inputs)
        out_tensor = get_raw_tensor(raw_out)

    emb = out_tensor.cpu().numpy().reshape(1, -1).astype("float32")
    faiss.normalize_L2(emb)   # in-place normalisation for cosine via FAISS L2
    return emb


# ---------------------------------------------------------------------------
# Stage 2 – Phi-4-mini identity probing via Ollama
# ---------------------------------------------------------------------------

def get_phi4_identity(path: str) -> str:
    """
    Ask phi4-mini to describe the true nature of the object in the image.

    Ollama's Python SDK sends vision content through `ollama.chat()` using
    the `images` key inside a message dict.  The SDK accepts either a file
    path string or raw bytes; base64 is NOT required when passing a path.
    """
    prompt = (
        "Is this a real object or a visual distractor such as a prop or ornament? "
        "Describe its true nature briefly in one or two sentences."
    )

    try:
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                    "images": [path],   # Ollama SDK accepts a file path directly
                }
            ],
        )
        # ollama.chat returns a dict; message content lives here:
        return response["message"]["content"].strip()

    except ollama.ResponseError as e:
        # Friendly hint if the model hasn't been pulled yet
        if "not found" in str(e).lower() or "pull" in str(e).lower():
            return (
                f"[Model not found] Run `ollama pull {OLLAMA_MODEL}` "
                f"and try again. Raw error: {e}"
            )
        return f"[Ollama ResponseError] {e}"

    except Exception as e:
        return f"[Ollama Error] {e}"


initialize_stack()


--- [INFO] Initializing Stack on: cpu ---


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- [SUCCESS] Models ready. CLIP dim: 512 ---


In [ ]:
# ---------------------------------------------------------------------------
# Main HSVF pipeline
# ---------------------------------------------------------------------------

def run_hsvf_pipeline(query_path: str, gallery_dir: str) -> None:
    print(f"\n{'=' * 20} HSVF RETRIEVAL {'=' * 20}")

    # Convert to absolute paths for robustness
    query_path = os.path.abspath(query_path)
    gallery_dir = os.path.abspath(gallery_dir)

    # Validate query image exists
    if not os.path.isfile(query_path):
        print(f"[ERROR] Query image not found: {query_path}")
        return

    # Validate gallery directory exists
    if not os.path.isdir(gallery_dir):
        print(f"[ERROR] Gallery directory not found: {gallery_dir}")
        return

    print(f"[INFO] Query: {query_path}")
    print(f"[INFO] Gallery: {gallery_dir}\n")

    # Collect gallery images
    exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
    gallery_paths = [
        os.path.join(gallery_dir, f)
        for f in os.listdir(gallery_dir)
        if os.path.isfile(os.path.join(gallery_dir, f)) and os.path.splitext(f.lower())[1] in exts
    ]

    if not gallery_paths:
        print(f"[WARNING] No valid images found in: {gallery_dir}")
        print(f"[INFO] Supported extensions: {exts}")
        return

    print(f"[INFO] Found {len(gallery_paths)} gallery images")

    # ------------------------------------------------------------------
    # Stage 1 – Build FAISS index over the gallery
    # ------------------------------------------------------------------
    index = faiss.IndexFlatL2(MODELS["dim"])
    for i, p in enumerate(gallery_paths):
        try:
            emb = get_clip_features(p)   # already normalised
            index.add(emb)
        except Exception as e:
            print(f"[WARNING] Failed to process {os.path.basename(p)}: {e}")
            continue

    # Retrieve all gallery images ranked by visual similarity
    q_emb = get_clip_features(query_path)   # already normalised
    distances, indices = index.search(q_emb, len(gallery_paths))

    # ------------------------------------------------------------------
    # Stage 2 – Query identity via Phi-4-mini
    # ------------------------------------------------------------------
    q_identity = get_phi4_identity(query_path)
    print(f"QUERY IDENTITY (phi4-mini): {q_identity[:120]}\n")

    q_text_emb = MODELS["text_model"].encode(q_identity, convert_to_tensor=True)

    # ------------------------------------------------------------------
    # Stage 3 – Per-candidate: identity probe + semantic fusion
    # ------------------------------------------------------------------
    results = []
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        img_path = gallery_paths[idx]
        img_identity = get_phi4_identity(img_path)

        # Semantic similarity between query and candidate descriptions
        m_text_emb = MODELS["text_model"].encode(img_identity, convert_to_tensor=True)
        semantic_score = util.cos_sim(q_text_emb, m_text_emb).item()

        # After L2-normalisation, FAISS L2 distance d satisfies:
        #   cosine_sim = 1 - d/2   (d in [0, 2])
        # We convert to a [0, 1] visual score:
        visual_score = 1.0 - dist / 2.0          # equivalent to cosine similarity
        visual_score = float(np.clip(visual_score, 0.0, 1.0))

        final_score = ALPHA * visual_score + (1 - ALPHA) * semantic_score

        results.append(
            {
                "filename": os.path.basename(img_path),
                "v_rank": rank + 1,
                "score": final_score,
                "visual_score": visual_score,
                "semantic_score": semantic_score,
                "desc": img_identity,
            }
        )

    # ------------------------------------------------------------------
    # Output re-ranked table
    # ------------------------------------------------------------------
    results.sort(key=lambda x: x["score"], reverse=True)

    col = {"fn": 22, "vr": 8, "nr": 8, "sc": 8, "vs": 8, "ss": 8}
    header = (
        f"{'FILENAME':<{col['fn']}} | {'V-RANK':<{col['vr']}} | "
        f"{'NEW RANK':<{col['nr']}} | {'SCORE':<{col['sc']}} | "
        f"{'V-SCORE':<{col['vs']}} | {'S-SCORE':<{col['ss']}} | PHI-4 REASONING"
    )
    print(header)
    print("-" * 130)
    for i, res in enumerate(results):
        desc_preview = res["desc"][:60].replace("\n", " ")
        print(
            f"{res['filename']:<{col['fn']}} | "
            f"{res['v_rank']:<{col['vr']}} | "
            f"{i + 1:<{col['nr']}} | "
            f"{res['score']:.4f}   | "
            f"{res['visual_score']:.4f}   | "
            f"{res['semantic_score']:.4f}   | "
            f"{desc_preview}..."
        )


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # Use os.path.join for cross-platform path handling
    query_path = "query_apple.jpg"
    gallery_dir = "."
    run_hsvf_pipeline(query_path, gallery_dir)



==================== HSVF RETRIEVAL ====================
[INFO] Query: /content/query_apple.jpg
[INFO] Gallery: /content

[INFO] Found 5 gallery images
QUERY IDENTITY (phi4-mini): [Ollama Error] Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://olla

FILENAME               | V-RANK   | NEW RANK | SCORE    | V-SCORE  | S-SCORE  | PHI-4 REASONING
----------------------------------------------------------------------------------------------------------------------------------
query_apple.jpg        | 1        | 1        | 1.0000   | 1.0000   | 1.0000   | [Ollama Error] Failed to connect to Ollama. Please check tha...
apple1.jpg             | 2        | 2        | 0.9885   | 0.9712   | 1.0000   | [Ollama Error] Failed to connect to Ollama. Please check tha...
apple2.jpg             | 3        | 3        | 0.9604   | 0.9011   | 1.0000   | [Ollama Error] Failed to connect to Ollama. Please check tha...
apple4.jpg             | 4        | 4 